In [73]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [74]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv")

In [75]:
sample.tail()

,id,health_condition
295748,985836,at-risk
295749,985837,at-risk
295750,985838,at-risk
295751,985839,at-risk
295752,985840,at-risk


In [76]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [77]:
train.shape

(690088, 15)

In [78]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [79]:
train.isna().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [80]:
num_cols = train.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    train[col]=train[col].fillna(train[col].median())

In [81]:
cat_cols = train.select_dtypes(include=['object']).columns

for col in cat_cols:
    train[col] = train[col].fillna('Unknown')

In [82]:
train.isna().sum()

id                         0
health_condition           0
sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

In [83]:
for col in train.select_dtypes(include='object').columns:
    print(col,train[col].nunique())

health_condition 3
diet_type 4
stress_level 4
sleep_quality 4
physical_activity_level 4
smoking_alcohol 4
gender 4


all these cols have not that much categories so we don't need to drop it.

In [84]:
train.drop(columns='id',inplace=True)

In [85]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           690088 non-null  float64
 2   heart_rate               690088 non-null  float64
 3   bmi                      690088 non-null  float64
 4   calorie_expenditure      690088 non-null  float64
 5   step_count               690088 non-null  float64
 6   exercise_duration        690088 non-null  float64
 7   water_intake             690088 non-null  float64
 8   diet_type                690088 non-null  object 
 9   stress_level             690088 non-null  object 
 10  sleep_quality            690088 non-null  object 
 11  physical_activity_level  690088 non-null  object 
 12  smoking_alcohol          690088 non-null  object 
 13  gender                   690088 non-null  object 
dtypes: f

In [86]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include=["object"]).columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    encoders[col] = le

In [87]:
for col, le in encoders.items():
    print(f"\nColumn: {col}")
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(mapping)


Column: health_condition
{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}

Column: diet_type
{'Unknown': np.int64(0), 'balanced': np.int64(1), 'non-veg': np.int64(2), 'veg': np.int64(3)}

Column: stress_level
{'Unknown': np.int64(0), 'high': np.int64(1), 'low': np.int64(2), 'medium': np.int64(3)}

Column: sleep_quality
{'Unknown': np.int64(0), 'average': np.int64(1), 'good': np.int64(2), 'poor': np.int64(3)}

Column: physical_activity_level
{'Unknown': np.int64(0), 'active': np.int64(1), 'moderate': np.int64(2), 'sedentary': np.int64(3)}

Column: smoking_alcohol
{'Unknown': np.int64(0), 'no': np.int64(1), 'occasional': np.int64(2), 'yes': np.int64(3)}

Column: gender
{'Unknown': np.int64(0), 'female': np.int64(1), 'male': np.int64(2), 'other': np.int64(3)}


In [88]:
train.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,2,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,3,1,1,3,3,1
1,0,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,2,2,1,2,3,3
2,2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,3,1,3,1,3,2
3,2,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,3,1,1,1,2,1
4,0,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,3,0,1,3,0,2


## adding new features

In [89]:
train["stress_smoking"] = (
    train["stress_level"] *
    train["smoking_alcohol"]
)

train["healthy_lifestyle"] = (
    train["diet_type"] *
    train["physical_activity_level"]
)

train["stress_sleep"] = (
    train["stress_level"] *
    train["sleep_duration"]
)

train["stress_sleep_quality"] = (
    train["stress_level"] *
    train["sleep_quality"]
)

train["activity_sleep"] = (
    train["physical_activity_level"] *
    train["sleep_duration"]
)

train["diet_sleep"] = (
    train["diet_type"] *
    train["sleep_quality"]
)

train["stress_diet"] = (
    train["stress_level"] *
    train["diet_type"]
)

## Models

In [90]:
X = train.drop('health_condition',axis=1)
y = train['health_condition']

In [91]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [92]:
print(X_train.shape)
print(X_val.shape)

(552070, 20)
(138018, 20)


## XGBoost

In [93]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import StratifiedKFold, cross_val_score

# XGBmodel = XGBClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     eval_metric='mlogloss'
# )

In [94]:
# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# scores = cross_val_score(
#     XGBmodel,
#     X,
#     y,
#     cv=cv,
#     scoring='accuracy',
#     n_jobs=-1
# )

# print("Fold Accuracies:", scores)
# print("Mean Accuracy:", scores.mean())
# print("Standard Deviation:", scores.std())

In [95]:
# XGBmodel.fit(X, y)

## LightBGM

In [96]:
from lightgbm import LGBMClassifier

In [97]:
# from lightgbm import LGBMClassifier

# lgbm = LGBMClassifier(
#     objective="multiclass",
#     num_class=3,

#     boosting_type="gbdt",

#     n_estimators=700,
#     learning_rate=0.04,

#     num_leaves=63,
#     max_depth=10,

#     min_child_samples=20,

#     subsample=0.8,
#     subsample_freq=1,

#     colsample_bytree=0.8,

#     reg_alpha=0.1,
#     reg_lambda=1.0,

#     random_state=42,
#     n_jobs=-1
# )
# lgbm.fit(X_train, y_train)

In [98]:
# pred = lgbm.predict(X_val)

In [99]:
# from sklearn.metrics import balanced_accuracy_score

# score = balanced_accuracy_score(y_val, pred)
# print(score)

In [100]:
# importance = pd.DataFrame({
#     "Feature": X_train.columns,
#     "Importance": model.feature_importances_
# })

# importance = importance.sort_values(by="Importance", ascending=False)

# print(importance.head(20))

## RandomForestClassifier

In [101]:
# from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(
#     n_estimators=200,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X_train,y_train)

In [102]:
# pred = rf.predict(X_val)

In [103]:
# from sklearn.metrics import balanced_accuracy_score

# score = balanced_accuracy_score(y_val, pred)

# print("Balanced Accuracy:", score)

In [104]:
# feature_importance = (
#     pd.DataFrame({
#         'feature': X.columns,
#         'importance': rf.feature_importances_
#     })
#     .sort_values('importance', ascending=False)
# )

# feature_importance.tail(10)

## Test 

In [105]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

In [106]:
test.shape

(295753, 14)

In [107]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [108]:
test.isna().sum()

id                             0
sleep_duration             32571
heart_rate                  3357
bmi                         5956
calorie_expenditure        22652
step_count                  5964
exercise_duration           2958
water_intake               18633
diet_type                   2958
stress_level               35490
sleep_quality              24999
physical_activity_level    15695
smoking_alcohol            12249
gender                      9160
dtype: int64

In [109]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

for col in num_cols:
    test[col]= test[col].fillna(X[col].median())

for col in cat_cols:
    test[col] = test[col].fillna('Unknown')

KeyError: 'stress_smoking'

In [ ]:
test.isna().sum().sum()

In [ ]:
test.head()

In [ ]:
student_ids = test['id'].copy()

In [ ]:
test.drop(columns='id',inplace=True)

In [ ]:
# diet_type
test["diet_type"] = test["diet_type"].map({
    "Unknown": 0,
    "balanced": 1,
    "non-veg": 2,
    "veg": 3
})

# stress_level
test["stress_level"] = test["stress_level"].map({
    "Unknown": 0,
    "high": 1,
    "low": 2,
    "medium": 3
})

# sleep_quality
test["sleep_quality"] = test["sleep_quality"].map({
    "Unknown": 0,
    "average": 1,
    "good": 2,
    "poor": 3
})

# physical_activity_level
test["physical_activity_level"] = test["physical_activity_level"].map({
    "Unknown": 0,
    "active": 1,
    "moderate": 2,
    "sedentary": 3
})

# smoking_alcohol
test["smoking_alcohol"] = test["smoking_alcohol"].map({
    "Unknown": 0,
    "no": 1,
    "occasional": 2,
    "yes": 3
})

# gender
test["gender"] = test["gender"].map({
    "Unknown": 0,
    "female": 1,
    "male": 2,
    "other": 3
})

In [ ]:
test["stress_smoking"] = (
    test["stress_level"] *
    test["smoking_alcohol"]
)

test["healthy_lifestyle"] = (
    test["diet_type"] *
    test["physical_activity_level"]
)

test["stress_sleep"] = (
    test["stress_level"] *
    test["sleep_duration"]
)

test["stress_sleep_quality"] = (
    test["stress_level"] *
    test["sleep_quality"]
)

test["activity_sleep"] = (
    test["physical_activity_level"] *
    test["sleep_duration"]
)

test["diet_sleep"] = (
    test["diet_type"] *
    test["sleep_quality"]
)

test["stress_diet"] = (
    test["stress_level"] *
    test["diet_type"]
)

In [ ]:
test_pred = lgbm.predict(test)

In [ ]:
test_pred[:10]

In [ ]:
submission = pd.DataFrame({
    "id": student_ids,
    "health_condition": test_pred
})

In [ ]:
submission.head(10)

In [ ]:
submission.isna().sum()

Column: health_condition
{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}

In [ ]:
submission["health_condition"] = submission["health_condition"].astype(int)

mapping = {
    0: "at-risk",
    1: "fit",
    2: "unhealthy"
}

submission["health_condition"] = submission["health_condition"].map(mapping)

In [ ]:
submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)